In [ ]:
import numpy as np
import pandas as pd
from word2vec_training import *
from emotion_classifier import *
from baseline_comparison import *
from embedding_analysis import *
from gensim.models import Word2Vec


In [ ]:
documents = load_documents(
    "./data/raw/",
    text_column="text"
)

In [ ]:
sentences = tokenize_documents(documents)

print(sentences[:5])

In [ ]:
vocabulary, word_to_index, index_to_word = build_vocabulary(
    sentences,
    min_count=5
)

save_vocabulary(
    word_to_index,
    index_to_word,
    "./models/"
)

In [ ]:
X, y = generate_training_data(
    sentences,
    word_to_index,
    window=5,
    architecture="skipgram"
)

print("Input examples:", X[:10])
print("Target examples:", y[:10])

In [ ]:
model = train_word2vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,
    epochs=10,
    negative=5,
    model_path="./models/word2vec_custom.model"
)

In [ ]:
results = run_hyperparameter_experiments(
    sentences,
    vector_sizes=[50, 100, 200, 300],
    windows=[2, 5, 10],
    architectures=["skipgram", "cbow"],
    epoch_values=[5, 10, 20]
)

results.head()

In [ ]:
find_similar_words(model, "doctor", topn=10)

In [ ]:
word_similarity(model, "hospital", "clinic")

In [ ]:
word_analogy(
    model,
    positive=["king", "woman"],
    negative=["man"],
    topn=5
)

In [ ]:
plot_word_embeddings(
    model,
    output_path="./outputs/figures/word2vec_tsne.png",
    top_n=500
)

In [ ]:
embedding_matrix = save_embedding_matrix(
    model,
    word_to_index,
    "./models/word2vec_embedding_matrix.npy"
)

In [ ]:
evaluation = evaluate_word_similarity(
    model,
    "./data/raw/wordsim353.csv"
)

In [ ]:
df, emotion_names, labels = load_goemotions(
    "./data/raw/",
    text_column="text",
    label_column="labels"
)

df.head()

In [ ]:
# Emotion distribution
plot_emotion_distribution(labels, emotion_names)

# Emotion counts
emotion_counts = get_emotion_counts(labels, emotion_names)

emotion_counts.to_csv(
    "./outputs/tables/emotion_counts.csv",
    index=False
)

display(emotion_counts)

In [ ]:
# Comment length distribution
length_stats = plot_comment_lengths(df["text"])

print(length_stats)

In [ ]:
# Display 5 examples per emotion
show_emotion_examples(
    df["text"],
    labels,
    emotion_names,
    number=5
)

In [ ]:
cleaned_texts = clean_emotion_texts(df["text"])

df["cleaned_text"] = cleaned_texts

df.to_csv(
    "./data/processed/goemotions_cleaned.csv",
    index=False
)

In [ ]:
x_train, x_val, x_test, y_train, y_val, y_test = split_data(
    cleaned_texts,
    labels
)

In [ ]:
tokenizer = prepare_tokenizer(
    x_train,
    num_words=30000
)

max_length = 100

x_train_pad = convert_to_sequences(
    tokenizer, x_train, max_length
)

x_val_pad = convert_to_sequences(
    tokenizer, x_val, max_length
)

x_test_pad = convert_to_sequences(
    tokenizer, x_test, max_length
)

save_tokenizer_vocabulary(
    tokenizer,
    "./models/tokenizer_word_index.json"
)

In [ ]:
glove_embeddings = load_glove(
    "./data/raw/glove.6B.100d.txt"
)

In [ ]:
coverage = check_glove_coverage(
    tokenizer,
    glove_embeddings
)

In [ ]:
embedding_matrix, coverage = create_embedding_matrix(
    tokenizer,
    glove_embeddings,
    embedding_dim=100,
    output_path="./models/glove_embedding_matrix.npy"
)

In [ ]:
num_classes = len(emotion_names)

dense_model = build_dense_model(
    embedding_matrix,
    max_length=100,
    trainable=False,
    num_classes=num_classes
)

dense_model = compile_model(
    dense_model,
    learning_rate=0.001,
    multi_label=True
)

dense_model.summary()

In [ ]:
sample_weights = create_sample_weights(y_train)

dense_history = train_model(
    dense_model,
    x_train_pad,
    y_train,
    x_val_pad,
    y_val,
    epochs=30,
    batch_size=32,
    sample_weight=sample_weights
)

In [ ]:
save_classifier(
    dense_model,
    "./models/emotion_classifier_dense.keras"
)

plot_training_history(
    dense_history,
    model_name="dense_model"
)

In [ ]:
cnn_model = build_cnn_model(
    embedding_matrix,
    max_length=100,
    trainable=False,
    num_classes=num_classes
)

cnn_model = compile_model(
    cnn_model,
    learning_rate=0.001,
    multi_label=True
)

cnn_history = train_model(
    cnn_model,
    x_train_pad,
    y_train,
    x_val_pad,
    y_val,
    epochs=30,
    batch_size=32,
    sample_weight=sample_weights
)

In [ ]:
save_classifier(
    cnn_model,
    "./models/emotion_classifier_cnn.keras"
)

plot_training_history(
    cnn_history,
    model_name="cnn_model"
)

In [ ]:
dense_results, dense_prob, dense_pred = evaluate_multilabel_model(
    dense_model,
    x_test_pad,
    y_test
)

cnn_results, cnn_prob, cnn_pred = evaluate_multilabel_model(
    cnn_model,
    x_test_pad,
    y_test
)

In [ ]:
save_evaluation_results(
    dense_results,
    model_name="Dense"
)

save_evaluation_results(
    cnn_results,
    model_name="CNN"
)

In [ ]:
tfidf_vectorizer, x_train_tfidf, x_val_tfidf, x_test_tfidf = create_tfidf_features(
    x_train,
    x_val,    
    x_test,    
    max_features=30000,    
    ngram_range=(1, 2)
)

In [ ]:
lr_model, lr_predictions, lr_results = train_tfidf_logistic_regression( 
    x_train_tfidf,
    y_train,    
    x_test_tfidf,    
    y_test
)

In [ ]:
save_baseline_model(lr_model, "./models/tfidf_logistic_regression.joblib")
save_tfidf_vectorizer(tfidf_vectorizer,"./models/tfidf_vectorizer.joblib")

In [ ]:
rf_model, rf_predictions, rf_results = train_tfidf_random_forest(
    x_train_tfidf,
    y_train,
    x_test_tfidf,
    y_test,
    n_estimators=100
)

In [ ]:
save_baseline_model(
    rf_model,
    "./models/tfidf_random_forest.joblib"
)

In [ ]:
word2vec_model = Word2Vec.load(
    "./models/word2vec_custom.model"
)

In [ ]:
word2vec_matrix, word2vec_coverage = create_word2vec_embedding_matrix(
    tokenizer,
    word2vec_model,
    output_path="./models/word2vec_emotion_embedding_matrix.npy"
)

In [ ]:
word2vec_classifier = build_embedding_classifier(
    word2vec_matrix,
    max_length=100,
    num_classes=len(emotion_names),
    trainable=False
)

word2vec_classifier.summary()

In [ ]:
word2vec_history = train_embedding_classifier(
    word2vec_classifier,
    x_train_pad,
    y_train,
    x_val_pad,
    y_val,
    epochs=30,
    batch_size=32
)

In [ ]:
word2vec_predictions, word2vec_probabilities, word2vec_results = evaluate_embedding_classifier(
    word2vec_classifier,
    x_test_pad,
    y_test,
    "Word2Vec"
)

In [ ]:
word2vec_classifier.save(
    "./models/emotion_classifier_word2vec.keras"
)

In [ ]:
glove_classifier = build_embedding_classifier(
    embedding_matrix,
    max_length=100,
    num_classes=len(emotion_names),
    trainable=False
)

glove_history = train_embedding_classifier(
    glove_classifier,
    x_train_pad,
    y_train,
    x_val_pad,
    y_val,
    epochs=30,
    batch_size=32
)

In [ ]:
glove_predictions, glove_probabilities, glove_results = evaluate_embedding_classifier(
    glove_classifier,
    x_test_pad,
    y_test,
    "GloVe"
)

In [ ]:
glove_classifier.save(
    "./models/emotion_classifier_glove.keras"
)

In [ ]:
all_results = [
    lr_results,
    rf_results,
    word2vec_results,
    glove_results
]

comparison_df = compare_model_results(
    all_results,
    "./outputs/tables/model_comparison.csv"
)

display(comparison_df)

In [ ]:
plot_model_comparison(
    comparison_df,
    "./outputs/figures/model_comparison.png"
)

In [ ]:
word2vec_model = Word2Vec.load(
    "./models/word2vec_custom.model"
)

In [ ]:
words, vectors, categories = get_emotion_word_vectors(
    word2vec_model,
    embedding_type="word2vec"
)

print(words)
print(vectors.shape)

In [ ]:
tsne_2d = plot_emotion_words_tsne(
    words,
    vectors,
    categories,
    dimensions=2,
    output_path="./outputs/figures/emotion_words_tsne_2d.png"
)

In [ ]:
tsne_3d = plot_emotion_words_tsne(
    words,
    vectors,
    categories,
    dimensions=3,
    output_path="./outputs/figures/emotion_words_tsne_3d.png"
)

In [ ]:
words_glove, vectors_glove, categories_glove = get_emotion_word_vectors(
    glove_embeddings,
    embedding_type="glove"
)

plot_emotion_words_tsne(
    words_glove,
    vectors_glove,
    categories_glove,
    dimensions=2,
    output_path="./outputs/figures/glove_emotion_tsne.png"
)

In [ ]:
result1 = embedding_analogy(
    word2vec_model,
    positive=["happy", "sadness"],
    negative=["joy"],
    topn=5
)

print(result1)

In [ ]:
result2 = embedding_analogy(
    word2vec_model,
    positive=["love", "hate"],
    negative=["like"],
    topn=5
)

print(result2)

In [ ]:
similarity_results = get_intensity_and_opposition(
    word2vec_model,
    embedding_type="word2vec"
)

display(similarity_results)

In [ ]:
similarity_results.to_csv(
    "./outputs/tables/emotion_word_similarity.csv",
    index=False
)

In [ ]:
attention_classifier, attention_model = build_attention_model(
    embedding_matrix,
    max_length=100,
    num_classes=len(emotion_names)
)

In [ ]:
attention_history = train_model(
    attention_classifier,
    x_train_pad,
    y_train,
    x_val_pad,
    y_val,
    epochs=30,
    batch_size=32
)

In [ ]:
sample_text = x_test[0]

attention_results = plot_attention_for_text(
    sample_text,
    tokenizer,
    attention_model,
    max_length=100,
    output_path="./outputs/figures/attention_heatmap.png"
)

In [ ]:
comment_vectors = create_comment_vectors(
    x_train_pad,
    embedding_matrix
)

print(comment_vectors.shape)

In [ ]:
emotion_centroids = calculate_emotion_centroids(
    comment_vectors,
    y_train,
    emotion_names
)

print(emotion_centroids.shape)

In [ ]:
centroid_distances = calculate_centroid_distances(
    emotion_centroids,
    emotion_names
)

display(centroid_distances)

In [ ]:
centroid_distances.to_csv(
    "./outputs/tables/emotion_centroid_distances.csv"
)

In [ ]:
emotion_similarity = calculate_centroid_similarity(
    emotion_centroids,
    emotion_names
)

display(emotion_similarity)

In [ ]:
plot_emotion_similarity_matrix(
    emotion_similarity,
    output_path="./outputs/figures/emotion_similarity_matrix.png"
)

In [ ]:
centroid_points = plot_centroids_tsne(
    emotion_centroids,
    emotion_names,
    dimensions=2,
    output_path="./outputs/figures/emotion_centroids_tsne.png"
)

In [ ]:
# Generate confusion matrix
conf_matrix = plot_confusion_matrix_27(
    y_test,
    glove_probabilities,
    emotion_names,
    output_path="./outputs/figures/confusion_matrix_27x27.png"
)

print(conf_matrix.shape)

In [ ]:
confused_pairs = find_confused_emotion_pairs(
    conf_matrix,
    emotion_names,
    top_n=10
)

display(confused_pairs)

In [ ]:
confused_pairs.to_csv(
    "./outputs/tables/most_confused_emotions.csv",
    index=False
)

In [ ]:
first_pair = confused_pairs.iloc[0]

true_emotion = first_pair["true_emotion"]
predicted_emotion = first_pair["predicted_emotion"]

print("True emotion:", true_emotion)
print("Predicted emotion:", predicted_emotion)

In [ ]:
error_examples = get_misclassified_examples(
    x_test,
    y_test,
    glove_probabilities,
    emotion_names,
    true_emotion,
    predicted_emotion,
    number=10
)

display(error_examples)

In [ ]:
save_error_examples(
    error_examples,
    "./outputs/tables/error_examples.csv"
)

In [ ]:
length_analysis = analyze_performance_by_length(
    x_test,
    y_test,
    glove_probabilities
)

display(length_analysis)

In [ ]:
frequency_analysis = analyze_performance_by_frequency(
    y_test,
    glove_probabilities,
    emotion_names
)

display(frequency_analysis)

In [ ]:
complexity_analysis = analyze_lexical_complexity(
    x_test,
    y_test,
    glove_probabilities
)

display(complexity_analysis)